In [ ]:
import json
import os
import subprocess
from pathlib import Path
from typing import Any, overload, Mapping
from dataclasses import dataclass

from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
from langchain_core.tools import tool
from langchain_ollama import ChatOllama



In [ ]:
OLLAMA_BASE_URL = os.getenv(
    "OLLAMA_BASE_URL",
    "http://10.42.0.192:11434/",
)

MODEL_NAME = os.getenv(
    "OLLAMA_MODEL",
    "gpt-oss:20b",
)

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0.01,
    num_ctx=40960,
)

print(f"Using {MODEL_NAME} at {OLLAMA_BASE_URL}")

In [ ]:
@dataclass
class ToolEvent:
    iteration: int
    tool: str
    args: dict
    result: str

In [ ]:
events: list[ToolEvent] = []

In [ ]:
response = llm.invoke("Reply with exactly: Ollama connection works")
print(response.content)


In [ ]:
WORKSPACE = Path("./agent_websockets2")
WORKSPACE.mkdir(parents=True, exist_ok=True)

WORKSPACE = WORKSPACE.resolve()
print(WORKSPACE)

In [ ]:
def safe_path(relative_path: str) -> Path:
    """
    Resolve a user-provided path inside WORKSPACE.
    Prevents paths such as ../../etc/passwd.
    """
    path = (WORKSPACE / relative_path).resolve()

    if path != WORKSPACE and WORKSPACE not in path.parents:
        raise ValueError(f"Path escapes workspace: {relative_path}")

    return path

In [ ]:
@tool
def list_files() -> str:
    """List files and directories in the current project workspace."""
    entries = []

    for path in sorted(WORKSPACE.rglob("*")):
        relative = path.relative_to(WORKSPACE)
        if ".git" in relative.parts or "__pycache__" in relative.parts:
            continue

        suffix = "/" if path.is_dir() else ""
        entries.append(f"{relative}{suffix}")

    return "\n".join(entries) if entries else "(workspace is empty)"


In [ ]:
@tool
def read_file(path: str) -> str:
    """Read a UTF-8 text file from the project workspace."""
    file_path = safe_path(path)

    if not file_path.exists():
        return f"File does not exist: {path}"

    if not file_path.is_file():
        return f"Not a file: {path}"

    content = file_path.read_text(encoding="utf-8")

    # Avoid flooding the model context with very large files.
    max_chars = 30_000
    if len(content) > max_chars:
        content = content[:max_chars] + "\n...[truncated]"

    return content


In [ ]:
from importlib.metadata import version

for package in [
    "langchain",
    "langchain-core",
    "langchain-ollama",
]:
    print(package, version(package))

In [ ]:
@tool
def write_file(path: str, content: Any, overwrite: bool = False) -> str:
    """
    Write `content` to `path`.  
    - If the file exists and `overwrite` is False (default), the call is a no‑op
      and you get a short “file exists” message.
    - If `overwrite` is True, the existing file is simply replaced.
    - `content` may be:
        • a mapping → pretty‑printed JSON (unless it contains a single string)
        • a string → written verbatim (real newlines, no `\\n`)

    Returns a human‑readable status message.
    """

    file_path = safe_path(path)

    if isinstance(content, Mapping):
        if "content" in content and isinstance(content["content"], str):
            content_str = content["content"]
        else:
            content_str = json.dumps(content, indent=2, ensure_ascii=False)
    elif isinstance(content, str):
        content_str = content
    else:
        raise TypeError(f"Unsupported content type {type(content)}")

    if file_path.exists() and not overwrite:
        return (
            f"File already exists: {path}. "
            "Use `overwrite=True` to replace it or use `read_file` + "
            "`edit_file` to modify it."
        )

    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(content_str, encoding="utf-8")
    return (
        f"Created {len(content_str)} characters in "
        f"{file_path.relative_to(Path.cwd())}"
    )


In [ ]:
@tool
def edit_file(path: str, old_text: str, new_text: str) -> str:
    """
    Edit a UTF-8 text file by replacing one exact occurrence of old_text
    with new_text.

    The file must already exist. The replacement is intentionally limited
    to one occurrence so the agent cannot accidentally modify multiple
    unrelated sections.
    """
    file_path = safe_path(path)

    if not file_path.exists():
        return f"File does not exist: {path}"

    if not file_path.is_file():
        return f"Not a file: {path}"

    content = file_path.read_text(encoding="utf-8")

    occurrences = content.count(old_text)

    if occurrences == 0:
        return (
            f"Could not edit {path}: old_text was not found. "
            "Read the file again and use an exact text match."
        )

    if occurrences > 1:
        return (
            f"Could not edit {path}: old_text occurs {occurrences} times. "
            "Provide a larger, more specific old_text block."
        )

    updated_content = content.replace(old_text, new_text, 1)
    file_path.write_text(updated_content, encoding="utf-8")

    return (
        f"Edited {file_path.relative_to(WORKSPACE)}: "
        f"replaced {len(old_text)} characters with {len(new_text)} characters."
    )


In [ ]:
@tool
def run_command(command: str) -> str:
    """
    Run a non-interactive shell command inside the project workspace.
    Use this for formatting, tests, compilation, and inspection.
    """
    blocked_fragments = [
        "rm -rf",
        "shutdown",
        "reboot",
        "mkfs",
        "dd if=",
        ":(){",
        "curl | sh",
        "wget | sh",
    ]

    normalized = command.lower().replace(" ", "")
    for fragment in blocked_fragments:
        if fragment.replace(" ", "") in normalized:
            return f"Blocked potentially destructive command: {command}"

    try:
        result = subprocess.run(
            command,
            shell=True,
            cwd=WORKSPACE,
            capture_output=True,
            text=True,
            timeout=60,
            env={
                **os.environ,
                "PYTHONUNBUFFERED": "1",
            },
        )

        output = (
            f"exit_code: {result.returncode}\n"
            f"stdout:\n{result.stdout}\n"
            f"stderr:\n{result.stderr}"
        )

        if len(output) > 20_000:
            output = output[:20_000] + "\n...[output truncated]"

        return output

    except subprocess.TimeoutExpired:
        return "Command timed out after 60 seconds."
    except Exception as exc:
        return f"Command failed to run: {type(exc).__name__}: {exc}"


In [ ]:
TOOLS = [
    list_files,
    read_file,
    write_file,
    edit_file,
    run_command
]


TOOLS_BY_NAME = {tool.name: tool for tool in TOOLS}

for item in TOOLS:
    print(item.name)


In [ ]:
llm_with_tools = llm.bind_tools(TOOLS)


In [ ]:
response = llm_with_tools.invoke(
    "Use the list_files tool and report the files in the workspace."
)

print("content:", response.content)
print("tool calls:", response.tool_calls)

In [ ]:
SYSTEM_PROMPT = """
The Year is 2026, You are an expert web developer and AI assistant.  
Assume the user may not even be as up to date as you are on possibilities.
offer help. use the tooling to create examples.
You are not allowed to declare success without reproducing and recording evidence..  

Rules:
- Inspect existing files.
- Use write_file to create example implementaions.
- Use edit_file for targeted modifications to existing files, modify anything you need to in the workspace.
- Use read_file to inspect relevant files before editing them.
- When using edit_file, provide an exact old_text match and a precise new_text replacement.
- If an edit_file operation fails because old_text was not found or is ambiguous, read the file again before retrying.
- Use run_command for tests, formatters, linters, compilers, and basic inspection.
- Do not claim that code works unless you actually run an appropriate check.
- Keep generated code focused and maintainable.
- Ask for clarification only when the requirement is genuinely ambiguous.
- Do not delete or overwrite unrelated files.
- All paths must be relative to the project workspace.
- When providing any text, double‑escape backslashes: `\\\\` instead of `\\`.  
"""

In [ ]:
print(SYSTEM_PROMPT)

In [ ]:
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    ToolMessage,
)

def run_agent(
    user_request: str,
    max_iterations: int = 100,
    verbose: bool = False,
) -> str:
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_request),
    ]

    for iteration in range(max_iterations):
        if verbose:
            print(f"\n--- iteration {iteration + 1} ---")

        response = llm_with_tools.invoke(messages)
        messages.append(response)

        events.append(
            ToolEvent(
                iteration=iteration + 1,
                tool="response",
                args={},
                result=str(response)
            )
        )

        print(len(messages))
        tool_calls = response.tool_calls or []

        if verbose:
            if response.content:
                print("Assistant:", response.content)
            print("Tool calls:", tool_calls)

        # The model is finished when it returns no tool calls.
        if not tool_calls:
            return {"condition" : "no tool calls",
                    "final_response": response.content,
                    "iterations": iteration + 1,
                    "events": events}

        for tool_call in tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call.get("args", {})
            tool_call_id = tool_call["id"]

            selected_tool = TOOLS_BY_NAME.get(tool_name)

            if selected_tool is None:
                tool_result = f"Unknown tool: {tool_name}"
            else:
                try:
                    tool_result = selected_tool.invoke(tool_args)
                except Exception as exc:
                    tool_result = (
                        f"Tool error: {type(exc).__name__}: {exc}"
                    )
                    
            events.append(
                ToolEvent(
                    iteration=iteration + 1,
                    tool=tool_name,
                    args=tool_args,
                    result=str(tool_result)
                )
            )
            
            if verbose:
                print(f"Executing: {tool_name}({tool_args})")
                print(str(tool_result)[:2_000])
            else:
                print(f"Executing: {tool_name}()")

            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call_id,
                )
            )

    return {"condition" : f"Agent stopped after {max_iterations} iterations. The workspace may contain partial results.",
            "final_response": response.content,
            "iterations": iteration + 1,
            "events": events} 


In [ ]:
request = """
---

## 🎯 Target Output

```
root/
├── docker-compose.yml          # orchestrates services
├── nginx/
│   ├── Dockerfile
│   └── nginx.conf
├── chat-server/
│   ├── Dockerfile
│   ├── package.json
│   └── server.js
└── chat-client/
    ├── Dockerfile
    ├── package.json
    └── client.js
```

* **chat‑server** – Node.js WebSocket chat server (listens on port 8080)
* **chat‑client** – Simple Node client that connects, sends a message, and prints replies
* **nginx** – Acts as a reverse proxy in front of the WebSocket server
* **docker‑compose.yml** – Builds all services and starts them together

---

## 📝 Prompt Template

```
You are an expert full‑stack developer. Create the following artifact set **exactly** as described below:

1. **Node WebSocket chat server**
   - File: `chat-server/server.js`
   - Should use the `ws` library, listen on port 8080, echo back any message received from a client.
   - Include minimal error handling.

2. **Node WebSocket client**
   - File: `chat-client/client.js`
   - Connects to `ws://chat-server:8080` (service name from compose), sends a single text message (e.g., "Hello from client") and prints any message it receives to 
stdout.

3. **Dockerfiles**
   - `chat-server/Dockerfile`: Uses `node:20-alpine`, copies `package.json` and `server.js`, installs dependencies, exposes port 8080, runs `node server.js`.
   - `chat-client/Dockerfile`: Similar to server but exposes nothing; entrypoint runs `node client.js`.
   - `nginx/Dockerfile`: Uses `nginx:1.27-alpine`, copies `nginx.conf`, exposes port 80.

4. **Nginx config**
   - File: `nginx/nginx.conf`
   - Acts as a reverse proxy to `chat-server:8080` for WebSocket connections (`upgrade: websocket`).

5. **docker‑compose.yml** (top‑level)
   - Three services: `nginx`, `chat-server`, `chat-client`.
   - Use `depends_on` so that nginx starts after chat‑server.
   - Expose `80` on host for the nginx service only.

Output **only** the following:
- A markdown list of the file paths (exactly as shown above) with a brief description of each file.
- The full content of each file, prefixed by `---` and the file path, like:
  ```
  --- chat-server/server.js
  <code>
  ```
  (No extra comments, no explanations after the code.)
- Do **not** include any other text, only the required files and a short list of paths.

Avoid asking for clarification. This is a single, complete request.
```

---

"""

result = run_agent(request)
print("\nFINAL RESPONSE\n")
print(result)


In [ ]:
request = """
> You are a coding agent that can modify files on the host via the tool `write_file(file_path: str, content: str)`.  
> Your task is to add a simple “chat box” on the web page so that:
> 1. A user can type text into an input field and click a button.  
> 2. The typed text is sent to the WebSocket server.  
> 3. The server broadcasts that text to **all other** connected clients.  
> 4. Every client (including the sender) prints the received message to the console.
> 
> **Update the following files** (no new files). For each file, call `write_file` with the exact path (relative to the repository root) and the full content shown below.

---

### 1. `chat-server/server.js`

```
const WebSocket = require('ws');
const http = require('http');

// HTTP health‑check (optional, harmless)
const httpServer = http.createServer((req, res) => {
  if (req.url === '/health') {
    res.writeHead(200, { 'Content-Type': 'text/plain' });
    res.end('ok');
  }
});
httpServer.listen(8081);

// WebSocket server
const wss = new WebSocket.Server({ port: 8080 });
const connections = []; // track all connected clients

wss.on('connection', (ws) => {
  console.log('Client connected');
  connections.push(ws);

  ws.on('message', (message) => {
    const text = typeof message === 'string' ? message : message.toString();
    console.log('Broadcasting:', text);

    // Send to every *other* client
    connections
      .filter((c) => c !== ws && c.readyState === WebSocket.OPEN)
      .forEach((c) => c.send(text));
  });

  ws.on('close', () => {
    console.log('Client disconnected');
    const idx = connections.indexOf(ws);
    if (idx !== -1) connections.splice(idx, 1);
  });

  ws.on('error', (err) => console.error('WebSocket error:', err));
});

console.log('WebSocket server listening on port 8080');
```

---

### 2. `nginx/static/client.js`

```
const WebSocket = require('ws');   // <‑‑ use ws package for Node; this file is served to browsers, but the same code works in the browser

const endpoint = process.env.ENDPOINT || 'ws1.mindbodyengineer.com';
const ws = new WebSocket(`wss://${endpoint}/ws/`);

ws.on('open', () => console.log('WebSocket connection opened'));

ws.on('message', (msg) => console.log('Received:', msg));

ws.on('error', (err) => console.error('WebSocket error:', err));

ws.on('close', () => console.log('Connection closed'));

// UI helpers – these functions are executed in the browser environment
function sendMessage() {
  const input = document.getElementById('msgInput');
  if (input && input.value.trim() !== '') {
    ws.send(input.value.trim());
    input.value = '';
  }
}

// Attach click handler when DOM is ready
document.addEventListener('DOMContentLoaded', () => {
  const btn = document.getElementById('sendBtn');
  if (btn) btn.addEventListener('click', sendMessage);
});
```

---

### 3. `nginx/static/index.html`

```
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>WebSocket Chat Demo</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 2rem; }
    #chat { margin-bottom: 1rem; }
    input[type="text"] { width: 70%; padding: 0.5rem; }
    button { padding: 0.5rem 1rem; }
    #log { border: 1px solid #ddd; padding: 1rem; height: 200px; overflow-y: auto; background: #f9f9f9; }
  </style>
</head>
<body>
  <h1>WebSocket Chat Demo</h1>

  <div id="chat">
    <input type="text" id="msgInput" placeholder="Type a message...">
    <button id="sendBtn">Send</button>
  </div>

  <div id="log"></div>

  <script src="client.js"></script>
  <script>
    // Append received messages to the log
    const logDiv = document.getElementById('log');
    const originalOnMessage = window.ws?.onmessage;
    window.ws = window.ws || {};
    window.ws.onmessage = (e) => {
      if (originalOnMessage) originalOnMessage(e);
      const msg = e.data;
      const p = document.createElement('p');
      p.textContent = msg;
      logDiv.appendChild(p);
      logDiv.scrollTop = logDiv.scrollHeight;
    };
  </script>
</body>
</html>


"""

result = run_agent(request)
print("\nFINAL RESPONSE\n")
print(result)


In [ ]:
result['condition']

In [ ]:
print(list_files.invoke({}))


In [ ]:
import pprint
pprint.pprint(events)

In [ ]:
print(read_file.invoke({"path": "README.md"}))


In [ ]:
print(run_command.invoke({
    "command": "python -m compileall app"
}))

In [ ]:
def interactive_agent():
    print("Coding agent ready.")
    print(f"Workspace: {WORKSPACE}")
    print("Type 'exit' or 'quit' to stop.\n")

    while True:
        try:
            request = input("You> ").strip()
        except EOFError:
            break

        if request.lower() in {"exit", "quit"}:
            break

        if not request:
            continue

        answer = run_agent(request, verbose=True)
        print(f"\nAgent> {answer}\n")
